# Extended Thinking

## Table of contents
- [Setup](#setup)
- [Basic example](#basic-example)
- [Streaming with extended thinking](#streaming-with-extended-thinking)
- [Token counting and context window management](#token-counting-and-context-window-management)
- [Understanding redacted thinking](#understanding-redacted-thinking-blocks)
- [Handling error cases](#handling-error-cases)

This notebook demonstrates how to use Claude's extended thinking feature on Claude Sonnet 5.5 with various examples and edge cases.

Extended thinking gives Claude enhanced reasoning capabilities for complex tasks, while also providing transparency into its step-by-step thought process before it delivers its final answer. When extended thinking is turned on, Claude creates `thinking` content blocks where it outputs its internal reasoning. Claude incorporates insights from this reasoning before crafting a final response.

On Claude Sonnet 5.5, thinking is **adaptive**: you pass `thinking={"type": "adaptive"}` (this is also the default when `thinking` is omitted) and the model decides how much to think for each request. You steer how hard it thinks with `output_config={"effort": ...}` (`"low"`, `"medium"`, `"high"`, `"xhigh"`, or `"max"`; the default is `"high"`) rather than with a token budget. The older `{"type": "enabled", "budget_tokens": N}` form is not accepted on this model. For more information on extended thinking, see our [documentation](https://platform.claude.com/docs/en/build-with-claude/thinking).

## Setup

First, let's install the necessary packages and set up our environment.

In [ ]:
%pip install anthropic

In [1]:
import anthropic

# Set your API key as an environment variable or directly
# os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

# Initialize the client
client = anthropic.Anthropic()


# Helper functions
def print_thinking_response(response):
    """Pretty print a message response with thinking blocks."""
    print("\n==== FULL RESPONSE ====")
    for block in response.content:
        if block.type == "thinking":
            print("\n🧠 THINKING BLOCK:")
            # Show truncated thinking for readability
            print(block.thinking[:500] + "..." if len(block.thinking) > 500 else block.thinking)
            print(f"\n[Signature available: {bool(getattr(block, 'signature', None))}]")
            if hasattr(block, "signature") and block.signature:
                print(f"[Signature (first 50 chars): {block.signature[:50]}...]")
        elif block.type == "redacted_thinking":
            print("\n🔒 REDACTED THINKING BLOCK:")
            print(f"[Data length: {len(block.data) if hasattr(block, 'data') else 'N/A'}]")
        elif block.type == "text":
            print("\n✓ FINAL ANSWER:")
            print(block.text)

    print("\n==== END RESPONSE ====")


def count_tokens(messages):
    """Count tokens for a given message list."""
    result = client.messages.count_tokens(model="claude-sonnet-5-5", messages=messages)
    return result.input_tokens

## Basic example

Let's start with a basic example to show extended thinking in action.

One detail to note before we start: on Claude Sonnet 5.5 the `display` setting for thinking defaults to `"omitted"`, which returns `thinking` blocks with empty text (you still get the signature, and you are still billed for the thinking tokens). Because this notebook prints the reasoning, every request sets `"display": "summarized"` so the API returns a summary of Claude's thinking.

Adaptive also means Claude can decide a prompt needs no thinking at all, in which case the response simply has no `thinking` block. That is most likely on easy prompts and at lower `effort`. The examples below use a small constraint-satisfaction puzzle and `"effort": "xhigh"` so you reliably see thinking; the token-counting section later shows how the amount of thinking changes as you move `effort` from `"low"` to `"xhigh"`.

In [2]:
def basic_thinking_example():
    response = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=16000,  # thinking tokens count toward max_tokens, so leave headroom
        thinking={
            "type": "adaptive",
            "display": "summarized",  # default is "omitted" (empty thinking text)
        },
        output_config={"effort": "xhigh"},  # low | medium | high (default) | xhigh | max
        messages=[
            {
                "role": "user",
                "content": "Five talks (A, B, C, D, E) must be scheduled in five consecutive slots. Constraints: A comes before C; B is not adjacent to D; E is neither first nor last; C is immediately before or after E; D comes before A. List every valid ordering and explain how you know there are no others.",
            }
        ],
    )

    print_thinking_response(response)


basic_thinking_example()


==== FULL RESPONSE ====

🧠 THINKING BLOCK:
I'm working through the constraints: D must come before A, which comes before C, E must sit adjacent to C and be in positions 2-4, and B can't be next to D. I'm now systematically trying placements of D to narrow down valid orderings.

Checking the other two candidates confirms all constraints hold: D before A, A before C, C adjacent E, and no B next to D. So the three valid orderings are DABEC, DAECB, and DACEB.



[Signature available: True]
[Signature (first 50 chars): CAQS8RIKEAgSGAI4AUIIdGhpbmtpbmcSDC9PApxA+kqk+axotR...]

✓ FINAL ANSWER:
**Valid orderings (slots 1→5):**

1. D A B E C
2. D A E C B
3. D A C E B

**Why there are no others**

1. Combining "D before A" and "A before C" gives the chain D < A < C.
2. E must be next to C, so E is either immediately before C (block "EC") or immediately after C (block "CE"). A cannot be E, so in either case A comes before the whole block. That leaves D, A, [block] in that fixed order, with B free 

## Streaming with extended thinking

This example shows how to handle streaming with thinking:

In [3]:
def streaming_with_thinking():
    with client.messages.stream(
        model="claude-sonnet-5-5",
        max_tokens=16000,
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": "xhigh"},
        messages=[
            {
                "role": "user",
                "content": "Five talks (A, B, C, D, E) must be scheduled in five consecutive slots. Constraints: A comes before C; B is not adjacent to D; E is neither first nor last; C is immediately before or after E; D comes before A. List every valid ordering and explain how you know there are no others.",
            }
        ],
    ) as stream:
        # Track what we're currently building
        current_block_type = None
        current_content = ""

        for event in stream:
            if event.type == "content_block_start":
                current_block_type = event.content_block.type
                print(f"\n--- Starting {current_block_type} block ---")
                current_content = ""

            elif event.type == "content_block_delta":
                if event.delta.type == "thinking_delta":
                    print(
                        event.delta.thinking, end="", flush=True
                    )  # Just print dots for thinking to avoid clutter
                    current_content += event.delta.thinking
                elif event.delta.type == "text_delta":
                    print(event.delta.text, end="", flush=True)
                    current_content += event.delta.text

            elif event.type == "content_block_stop":
                if current_block_type == "thinking":
                    # Just show a summary for thinking
                    print(f"\n[Completed thinking block, {len(current_content)} characters]")
                elif current_block_type == "redacted_thinking":
                    print("\n[Redacted thinking block]")
                print(f"--- Finished {current_block_type} block ---\n")
                current_block_type = None

            elif event.type == "message_stop":
                print("\n--- Message complete ---")


streaming_with_thinking()


--- Starting thinking block ---
I need to work through the constraints: D before A before C, E adjacent to C, and B not adjacent to D. I'm trying to build the ordering by placing D, A, C first then inserting E next to C and checking where B can go without violating the adjacency rule with D.

Testing "D A C E B" works since D and B aren't adjacent and E sits at position 4, not the ends. Now I'm checking the other case where E comes immediately before C, inserting B in each of the four gaps and verifying the adjacency and positional rules each time.

After going through DBAEC, DABEC, and DAECB, I find that DABEC and DAECB both satisfy every condition while DBAEC fails because B ends up next to D. Combined with the earlier valid case, that gives three total valid orderings: DACEB, DABEC, and DAECB.


[Completed thinking block, 777 characters]
--- Finished thinking block ---


--- Starting text block ---
**Valid orderings (3):**

1. D A C E B
2. D A B E C
3. D A E C B

**Why there are no

## Token counting and context window management

This example demonstrates how to track token usage with extended thinking, and how the `effort` setting changes how much Claude thinks:

In [4]:
def token_counting_example():
    # Define a function to create a sample prompt
    def create_sample_messages():
        messages = [
            {
                "role": "user",
                "content": "Five talks (A, B, C, D, E) must be scheduled in five consecutive slots. Constraints: A comes before C; B is not adjacent to D; E is neither first nor last; C is immediately before or after E; D comes before A. List every valid ordering and explain how you know there are no others.",
            }
        ]
        return messages

    # Count tokens without thinking
    base_messages = create_sample_messages()
    base_token_count = count_tokens(base_messages)
    print(f"Base token count (input only): {base_token_count}")

    # Make a request with thinking and check actual usage
    max_tokens = 16000
    response = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=max_tokens,
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": "xhigh"},
        messages=base_messages,
    )

    # usage.output_tokens counts the *full* thinking plus the final answer,
    # even though the thinking text we get back is only a summary.
    summary_chars = sum(
        len(block.thinking) for block in response.content if block.type == "thinking"
    )

    final_answer_tokens = sum(
        len(block.text.split()) * 1.3  # Rough estimate
        for block in response.content
        if block.type == "text"
    )

    print(f"\nInput tokens (from usage): {response.usage.input_tokens}")
    print(f"Output tokens (from usage, thinking + answer): {response.usage.output_tokens}")
    print(f"Estimated final answer tokens: ~{int(final_answer_tokens)}")
    print(
        f"So roughly ~{response.usage.output_tokens - int(final_answer_tokens)} output tokens went to thinking"
    )
    print(f"Thinking summary returned: {summary_chars} characters")
    print(
        f"Input tokens + max_tokens = {base_token_count + max_tokens} (this total must fit in the model's context window)"
    )

    # Demo with escalating effort levels: same prompt, different effort.
    # There is no token budget on current models; effort is the knob that
    # trades thinking depth for latency and cost ("max" exists too). At the
    # lower levels Claude may decide this problem needs no thinking at all,
    # in which case the response has no thinking block.
    for effort in ["low", "medium", "high", "xhigh"]:
        response = client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=max_tokens,
            thinking={"type": "adaptive", "display": "summarized"},
            output_config={"effort": effort},
            messages=base_messages,
        )
        summary_chars = sum(len(b.thinking) for b in response.content if b.type == "thinking")
        answer = " ".join(b.text for b in response.content if b.type == "text")
        print(f"\nWith effort={effort!r}:")
        print(f"Output tokens (thinking + answer): {response.usage.output_tokens}")
        print(f"Thinking summary length: {summary_chars} characters")
        print(f"Final answer preview: {answer[:120]}...")


# Uncomment to run the example
token_counting_example()

Base token count (input only): 102

Input tokens (from usage): 102
Output tokens (from usage, thinking + answer): 1595
Estimated final answer tokens: ~518
So roughly ~1077 output tokens went to thinking
Thinking summary returned: 457 characters
Input tokens + max_tokens = 16102 (this total must fit in the model's context window)

With effort='low':
Output tokens (thinking + answer): 1043
Thinking summary length: 0 characters
Final answer preview: Constraints:
1. A before C
2. B not adjacent to D
3. E in slot 2, 3, or 4
4. C adjacent to E
5. D before A

From 1 and 5...

With effort='medium':
Output tokens (thinking + answer): 794
Thinking summary length: 0 characters
Final answer preview: Constraints:
1. A before C
2. B not adjacent to D
3. E in slot 2, 3, or 4
4. C adjacent to E
5. D before A

From 1 and 5...

With effort='high':
Output tokens (thinking + answer): 1150
Thinking summary length: 423 characters
Final answer preview: **Valid orderings (slots 1–5):**

1. D A C E B
2. D A E 

## Understanding redacted thinking blocks

Occasionally Claude's internal reasoning will be flagged by safety systems. When this occurs, we encrypt some or all of the `thinking` block and return it to you as a `redacted_thinking` block. These redacted thinking blocks are decrypted when passed back to the API, allowing Claude to continue its response without losing context.

This example demonstrates working with redacted thinking blocks using a special test string that triggers them:

In [5]:
def redacted_thinking_example():
    # Using the special test string that triggers redacted thinking
    response = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=16000,
        thinking={"type": "adaptive", "display": "summarized"},
        messages=[
            {
                "role": "user",
                "content": "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB",
            }
        ],
    )

    # Identify redacted thinking blocks
    redacted_blocks = [block for block in response.content if block.type == "redacted_thinking"]
    thinking_blocks = [block for block in response.content if block.type == "thinking"]
    text_blocks = [block for block in response.content if block.type == "text"]
    print(response.content)
    print(f"Response includes {len(response.content)} total blocks:")
    print(f"- {len(redacted_blocks)} redacted thinking blocks")
    print(f"- {len(thinking_blocks)} regular thinking blocks")
    print(f"- {len(text_blocks)} text blocks")

    # Show data properties of redacted blocks
    if redacted_blocks:
        print("\nRedacted thinking blocks contain encrypted data:")
        for i, block in enumerate(redacted_blocks[:3]):  # Show first 3 at most
            print(f"Block {i + 1} data preview: {block.data[:50]}...")

    # Print the final text output
    if text_blocks:
        print("\nFinal text response:")
        print(text_blocks[0].text)


# Uncomment to run the example
redacted_thinking_example()

[RedactedThinkingBlock(data='CAQSlgcKEAgSGAI4AUIIdGhpbmtpbmcSDAvmF1hXsbzdc6DDlRoMIy9+bFB/la3LWaH1IjC+I73fnh6GNH2OQ+TidhopDIDmbIW4mmtPmzkCaQ8UTME/PoGMzIXaO/eVST+TaIoqswZLLsrmRNXVM/3MhNxyw8jVP+FUnkD7uGtVa42A2hGhP9K3EULP3DSzs1CO/iPrA/4LJChjLBrcwxnQEfHQW5zPbhujyTxBQtmM6jGXhZCD1O941b2NVBz35R2EPIcLMIO23k6F1p3g8UtucM8AvfbJ8qyO+JMK1/lAKsRjEuE+w0HUzYV1J526sVbUuqCYlHDsVnNHdVTPm802tuEZJo6NdR3On14JGpl4jmq0auTenXRSFCqgyiAefctT6uTFKzvzUYgYiaHGTxfxtHrN6/JPWwGhCBDPdqF8tr1dk4pzYhTY5a4cmmHGi4f++OMknWHM0Vp7OKWZZRbVvrbnm2Quwcv1gjDP2/+cUWyVg5urBpy39bU6mPNasiQkrPg0yjB/VpBLOjQ5EE+zAZILCEo4zLY7K2WD1+UHfmTi59+mfnDcqaFLjhVRK3mqY5Ut0C9IO5JFKYXyZGITljOhADq3amYPthG4ES+wgh3I4HAVNj9wo0kh2KiS7h1HDh20vF1o+Q4RpF9XPKL3OYcLPQDHxX7ZHzHLGzERnTsNGDQnxYAy95AAzn1lcnOk/rPDK+q+uqptheVgraDCxqXPtZrYyI/P7qlirMGbCoDmyi4IzAC4n1ej09Li0G0pGe8B9C49nSwn5Czn1L23/BSX7XLzddaf+/aEHOhAD7h7w+ru/NI0WBBmbyiXh8XrzRwtFmRfhhSrWAGOnFc2VtCHUijFz4XTGf6CNG0QLTw7XlT6fS+02kai2aNcIjqhhcD32cEmo8Hj9ZRDhWvfPJuHAAv9PVF14W9F+tFGSc6iW/PyQqMRRVDc4yBSUJv/va3UpBX

## Handling error cases

When using extended thinking, keep in mind:

1. **Effort, not budgets**: Current models have no `budget_tokens`. Use `output_config={"effort": ...}` to control how hard Claude thinks: start at the default (`"high"`), drop to `"medium"` or `"low"` when latency and cost matter more than depth, and reach for `"xhigh"` or `"max"` on the hardest problems. Sending the legacy `thinking={"type": "enabled", "budget_tokens": N}` form to Claude Sonnet 5.5 returns a 400 error.

2. **Incompatible features**: Non-default `temperature`, `top_p`, or `top_k` values are rejected on Claude Sonnet 5.5 (with or without thinking), and you cannot pre-fill assistant responses.

3. **Pricing and limits**: Thinking tokens count towards `max_tokens` and the context window, and are billed as output tokens (you are billed for the full thinking, not the summary you get back). They also count towards your rate limits. Keep `max_tokens` generous and use streaming for very large values.

For more details on extended thinking with tool use, see the "Extended Thinking with Tool Use" notebook.

In [6]:
def demonstrate_common_errors():
    # 1. Error from sending the legacy manual-budget form.
    #    Current models only accept {"type": "adaptive"}; non-default
    #    temperature / top_p / top_k values are rejected the same way.
    try:
        client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=4000,
            thinking={
                "type": "enabled",  # Legacy form, not accepted on this model
                "budget_tokens": 1024,
            },
            messages=[{"role": "user", "content": "Explain quantum computing."}],
        )
    except Exception as e:
        print(f"\nError with legacy budget_tokens thinking config: {e}")

    # 2. Error from exceeding context window
    try:
        # Create a very large prompt (large enough to overflow the context window)
        long_content = "Please analyze this text. " + "This is sample text. " * 300000

        client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=20000,  # This plus the long prompt will exceed context window
            thinking={"type": "adaptive"},
            messages=[{"role": "user", "content": long_content}],
        )
    except Exception as e:
        print(f"\nError from exceeding context window: {e}")


# Run the common error examples
demonstrate_common_errors()


Error with legacy budget_tokens thinking config: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '"thinking.type.enabled" is not supported for this model. Use "thinking.type.adaptive" and "output_config.effort" to control thinking behavior.'}, 'request_id': 'req_011Cfdh4WYGPktVi5kjF5NAt'}

Error from exceeding context window: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 1800065 tokens > 1000000 maximum'}, 'request_id': 'req_011Cfdh4XWok6x352LDwYdg6'}
